# Postprocess into rasters and vector datasets




In [1]:
from pathlib import Path

import geopandas as gpd
import geoutils as gu
import numpy as np
import pandas as pd
import rasterio as rio
import xdem
from osgeo import gdal, ogr, osr
import rasterstats
from osgeo_utils import gdal_calc


import subkart

In [2]:
nodata = 255
crs = "EPSG:25833"

In [3]:
classifier = subkart.utils.load_classifier()

## Post processing

In [9]:
fname = subkart.utils.to_filename(
    f"nisjedata-substrat-{classifier.__class__.__name__.lower()}", "norge", "latest", crs.split(":")[1]
)
predict_file = f"{fname}.tif"
prob_file = f"{fname}_probability.tif"

In [ ]:
gdal.UseExceptions()

prediction_files = [f"{r}_prediction.tif" for r in subkart.sources.REGIONS]
probability_files = [f"{r}_probability.tif" for r in subkart.sources.REGIONS]

subkart.utils.merge_rasters(prediction_files, predict_file, nodata=nodata)
subkart.utils.merge_rasters(probability_files, prob_file, nodata=-10000.0)


## Vectorize raw prediction and write final raw nisjedata-substrat vector

In [ ]:

subkart.vectorize.with_gdal(predict_file, "polygons_raw.gpkg", int(crs.split(":")[1]), nodata=255)

gdf_raw = gpd.read_file("polygons_raw.gpkg")
reverse_map = {v: k for k, v in subkart.labelling.BUNNTYPE_MAPPING.items()}
gdf_raw["BunnType"] = gdf_raw["DN"].map(reverse_map)

fname_raw = subkart.utils.to_filename(
    f"nisjedata-substrat-{classifier.__class__.__name__.lower()}-raw",
    "norge", "latest", gdf_raw.crs.to_epsg(),
)
gdf_raw.to_file(f"{fname_raw}.gpkg", driver="GPKG", layer="bunntyper")
gdf_raw.to_parquet(f"{fname_raw}.geo.parquet", compression="snappy")
subkart.utils.to_postgis(gdf_raw, fname_raw)


# Create processed prediction raster:
   1. Remap class 1 (blanding) → highest-probability class (0=løsbunn or 2=fastbunn) using gdal_calc
   2. Sieve filter: replace isolated single pixels with their largest neighbour

In [8]:
predict_file_remapped = f"{fname}_remapped_tmp.tif"
predict_file_processed = f"{fname}_processed.tif"

In [ ]:

gdal_calc.Calc(
    calc="numpy.where(A==1, numpy.where(B >= C, numpy.uint8(0), numpy.uint8(2)), A)",
    outfile=predict_file_remapped,
    A=predict_file,
    B=prob_file,
    B_band=1,
    C=prob_file,
    C_band=3,
    type="Byte",
    NoDataValue=nodata,
    hideNoData=True,
    creation_options=["COMPRESS=DEFLATE", "TILED=YES", "BLOCKXSIZE=512", "BLOCKYSIZE=512", "BIGTIFF=IF_SAFER"],
    overwrite=True,
)

# Pre-fill processed file with remapped data; SieveFilter only writes changed pixels
gdal.Translate(
    predict_file_processed,
    predict_file_remapped,
    creationOptions=["COMPRESS=DEFLATE", "TILED=YES", "BLOCKXSIZE=512", "BLOCKYSIZE=512", "BIGTIFF=IF_SAFER"],
)

ds_src = gdal.Open(predict_file_remapped, gdal.GA_ReadOnly)
ds_dst = gdal.Open(predict_file_processed, gdal.GA_Update)
src_band = ds_src.GetRasterBand(1)
dst_band = ds_dst.GetRasterBand(1)
gdal.SieveFilter(
    src_band,
    src_band.GetMaskBand(),
    dst_band,
    threshold=1,
    connectedness=8,
    callback=gdal.TermProgress_nocb,
)
ds_src = None
ds_dst = None

Path(predict_file_remapped).unlink()


0...10...20...30...40...50...60...70...80...90...100 - done in 00:00:19.
0...10...20...30...40...50...60...70...80...90...100 - done in 00:00:10.


## Create processed 1-band probability raster from the 3-band source

* class 0 (løsbunn, incl. remapped/sieved blanding) band 1 = P(class=0)
* class 2 (fastbunn)                            band 3 = P(class=2)

In [ ]:

prob_file_processed = f"{fname}_probability_processed.tif"

gdal_calc.Calc(
    calc="numpy.where(A==2, C, B)",
    outfile=prob_file_processed,
    A=predict_file_processed,
    B=prob_file,
    B_band=1,
    C=prob_file,
    C_band=3,
    type="Float16",
    NoDataValue=-10000.0,
    hideNoData=True,
    creation_options=["COMPRESS=DEFLATE", "TILED=YES", "BLOCKXSIZE=512", "BLOCKYSIZE=512", "BIGTIFF=IF_SAFER"],
    overwrite=True,
)


NameError: name 'prob_file' is not defined

## Vectorize processed prediction raster

In [ ]:
subkart.vectorize.with_grass(
    predict_file_processed, "polygons_processed.gpkg"
)

gdf = gpd.read_file("polygons_processed.gpkg")
reverse_map = {v: k for k, v in subkart.labelling.BUNNTYPE_MAPPING.items()}
gdf["BunnType"] = gdf["DN"].map(reverse_map)

# Compute mean probability per polygon from the processed 1-band probability raster
stats = rasterstats.zonal_stats(
    gdf,
    prob_file_processed,
    stats=["mean"],
    nodata=-10000,
)
gdf["probability"] = [s["mean"] for s in stats]

gdf.to_file(f"{fname}.gpkg", driver="GPKG", layer="bunntyper")
gdf.to_parquet(f"{fname}.geo.parquet", compression="snappy")
subkart.utils.to_postgis(gdf, fname)


Vectorizing nisjedata-substrat-xgbclassifier_norge_latest_25833_processed.tif (EPSG:25833) …


Importing raster map <raster>...
   0   3   6   9  12  15  18  21  24  27  30  33  36  39  42  45  48  51  54  57  60  63  66  69  72  75  78  81  84  87  90  93  96  99 100
Extracting areas...
   0   3   6   9  12  15  18  21  24  27  30  33  36  39  42  45  48  51  54  57  60  63  66  69  72  75  78  81  84  87  90  93  96  99 100
Writing areas...
   0   4   8  12  16  20  24  28  32  36  40  44  48  52  56  60  64  68  72  76  80  84  88  92  96 100
Building topology for vector map <vector@PERMANENT>...
Registering primitives...
Building areas...    300     400     500     600     700     800     900    1000    1100    1200    1300    1400    1500    1600    1700    1800    1900    2000    2100    2200    2300    2400    2500    2600    2700    2800    2900    3000    3100    3200    3300    3400    3500    3600    3700    3800    3900    4000    4100    4200    4300    4400    4500    4600    4700    4800    4900    5000    5100    5200    5300    5400    5500    5600    5700    58